# Stage 2 production window exploration

This notebook is read-only with respect to the completed Stage 2 production datasets. It inspects the phone and watch artifacts, creates exploratory tables under `output/tables/`, and saves figures under `output/figures/`. It does not rerun Stage 2, create windows, normalize data, train a model, or select a train/validation/test split.

In [ ]:
from pathlib import Path
import gzip
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

CODE_DIR = Path.cwd()
if CODE_DIR.name != "code":
    CODE_DIR = CODE_DIR / "code"
DATASETS = {
    "phones": CODE_DIR / "har_stage2_production_phones_v1",
    "watch": CODE_DIR / "har_stage2_production_watch_v1",
}
OUTPUT_DIR = CODE_DIR / "output"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CHANNELS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
TIMESTAMP_UNIT = "ns"
print("Dataset roots:")
for name, path in DATASETS.items():
    print(f"  {name}: {path.resolve()}")

## Artifact inventory and schema inspection

In [ ]:
def print_artifact_inventory(root):
    rows = []
    for path in sorted(root.iterdir()):
        rows.append({"dataset": root.name, "file": path.name, "bytes": path.stat().st_size, "size_mb": path.stat().st_size / 1e6})
    inventory = pd.DataFrame(rows)
    print(inventory.to_string(index=False))
    return inventory

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def first_jsonl_record(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        line = handle.readline()
    return json.loads(line) if line else {}

def inspect_dataset(name, root):
    print(f"\n{'=' * 80}\n{name.upper()} FILES\n{'=' * 80}")
    inventory = print_artifact_inventory(root)
    summary = read_json(root / "stage2_summary.json")
    manifest = read_json(root / "shard_manifest.json")
    checkpoint = read_json(root / "checkpoint.json")
    print("\nStage 2 summary fields:", list(summary))
    print(json.dumps(summary, indent=2))
    print("\nCheckpoint:", json.dumps(checkpoint, indent=2))
    print(f"\nShard manifest: {len(manifest['shards'])} shards")
    print(json.dumps(manifest["shards"][:3], indent=2))
    for filename in ["group_eligibility.jsonl.gz", "pair_diagnostics.jsonl.gz", "rejections.jsonl.gz", "repair_provenance.jsonl.gz"]:
        record = first_jsonl_record(root / filename)
        print(f"\n{filename} fields: {list(record)}")
        print("Example:", json.dumps(record, indent=2)[:2500])
    first_shard = sorted(root.glob("windows-*.npz"))[0]
    with np.load(first_shard, allow_pickle=False) as archive:
        print(f"\nNPZ example: {first_shard.name}")
        print("Keys:", archive.files)
        for key in archive.files:
            array = archive[key]
            values = array.reshape(-1)[:8] if array.ndim else np.asarray([array.item()])
            print(f"{key}: shape={array.shape}, dtype={array.dtype}, samples={values!r}")
        print("First X window, first 3 samples:\n", archive["X"][0, :3, :])
        example_metadata = json.loads(str(archive["metadata"][0]))
    print("Metadata fields:", sorted(example_metadata))
    print("Example metadata record:\n", json.dumps(example_metadata, indent=2))
    return {"inventory": inventory, "summary": summary, "manifest": manifest, "checkpoint": checkpoint}

artifact_info = {name: inspect_dataset(name, root) for name, root in DATASETS.items()}
print("\nDocumented channel order:", CHANNELS)
print("The production labels are already activity strings; no numeric label mapping is applied.")

In [ ]:
def load_production_dataset(name, root):
    arrays, labels, metadata, shard_rows = [], [], [], []
    for shard_path in sorted(root.glob("windows-*.npz")):
        with np.load(shard_path, allow_pickle=False) as archive:
            arrays.append(archive["X"])
            labels.append(archive["labels"])
            metadata.extend(json.loads(str(value)) for value in archive["metadata"])
            shard_rows.append({"dataset": name, "shard": shard_path.name, "windows": len(archive["labels"]), "bytes": shard_path.stat().st_size})
    X = np.concatenate(arrays, axis=0)
    label_array = np.concatenate(labels, axis=0)
    frame = pd.DataFrame(metadata)
    frame["dataset"] = name
    frame["label_from_npz"] = label_array
    frame["start_datetime"] = pd.to_datetime(frame["start_timestamp_raw"], unit=TIMESTAMP_UNIT, errors="coerce")
    frame["end_datetime"] = pd.to_datetime(frame["end_timestamp_raw"], unit=TIMESTAMP_UNIT, errors="coerce")
    frame["start_day"] = frame["start_datetime"].dt.date.astype("string")
    frame["start_hour"] = frame["start_datetime"].dt.hour
    return {"X": X, "labels": label_array, "metadata": frame, "shards": pd.DataFrame(shard_rows)}

datasets = {name: load_production_dataset(name, root) for name, root in DATASETS.items()}
metadata = pd.concat([value["metadata"] for value in datasets.values()], ignore_index=True)
metadata["dataset"] = pd.Categorical(metadata["dataset"], categories=["phones", "watch"])
X_combined = np.concatenate([datasets["phones"]["X"], datasets["watch"]["X"]], axis=0)
labels_combined = metadata["label_from_npz"].to_numpy()

overview = (metadata.groupby("dataset", observed=False).size().rename("accepted_windows").reset_index())
overview = pd.concat([overview, pd.DataFrame([{"dataset": "combined", "accepted_windows": len(metadata)}])], ignore_index=True)
overview.to_csv(TABLE_DIR / "dataset_overview_totals.csv", index=False)
print(overview.to_string(index=False))
print("\nShapes:", {name: value["X"].shape for name, value in datasets.items()}, "combined:", X_combined.shape)
label_mapping = pd.DataFrame({"activity_label": sorted(metadata["label_from_npz"].unique())})
label_mapping["label_value"] = label_mapping["activity_label"]
label_mapping.to_csv(TABLE_DIR / "activity_label_mapping.csv", index=False)
print("Participants:", metadata["User"].nunique(), "models:", metadata["Model"].nunique(), "activities:", metadata["label_from_npz"].nunique())
print("Groups:", metadata[["User", "Model", "Device"]].drop_duplicates().shape[0])
display(label_mapping)

## Dataset overview

In [ ]:
def save_count_table(group_columns, filename):
    table = metadata.groupby(group_columns, observed=False).size().rename("accepted_windows").reset_index()
    table.to_csv(TABLE_DIR / filename, index=False)
    display(table)
    return table

participant_counts = save_count_table(["dataset", "User"], "windows_by_participant.csv")
device_counts = save_count_table(["dataset", "Model"], "windows_by_device_model.csv")
activity_counts = save_count_table(["dataset", "label_from_npz"], "windows_by_activity.csv")
plt.figure(figsize=(7, 4))
overview[overview["dataset"] != "combined"].plot.bar(x="dataset", y="accepted_windows", legend=False, ax=plt.gca(), color=["#4472c4", "#ed7d31"])
plt.title("Total accepted windows by production dataset")
plt.xlabel("Dataset")
plt.ylabel("Accepted windows")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "overview_accepted_windows.png", dpi=160)
plt.show()

## Activity distribution

In [ ]:
activity_table = metadata.groupby(["dataset", "label_from_npz"], observed=False).size().rename("windows").reset_index()
activity_table["percentage"] = activity_table["windows"] / activity_table.groupby("dataset", observed=False)["windows"].transform("sum") * 100
activity_table.to_csv(TABLE_DIR / "activity_distribution.csv", index=False)
display(activity_table)

for metric, ylabel, suffix in [("windows", "Accepted windows", "counts"), ("percentage", "Percentage of dataset", "percentages")]:
    for dataset_name in ["phones", "watch"]:
        subset = activity_table[activity_table["dataset"].eq(dataset_name)].set_index("label_from_npz")[metric]
        ax = subset.reindex(sorted(activity_table["label_from_npz"].unique()), fill_value=0).plot.bar(figsize=(9, 4), title=f"{dataset_name.title()} activity distribution ({metric})", color="#4472c4" if dataset_name == "phones" else "#ed7d31")
        ax.set_xlabel("Recorded activity label")
        ax.set_ylabel(ylabel)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"activity_distribution_{dataset_name}_{suffix}.png", dpi=160)
        plt.show()
    combined = metadata.groupby("label_from_npz", observed=False).size()
    if metric == "percentage":
        combined = combined / len(metadata) * 100
    ax = combined.reindex(sorted(combined.index)).plot.bar(figsize=(9, 4), title=f"Combined activity distribution ({metric})", color="#70ad47")
    ax.set_xlabel("Recorded activity label")
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"activity_distribution_combined_{suffix}.png", dpi=160)
    plt.show()

## Participant differences

In [ ]:
participant_activity = metadata.groupby(["dataset", "User", "label_from_npz"], observed=False).size().rename("windows").reset_index()
participant_activity.to_csv(TABLE_DIR / "participant_activity_counts.csv", index=False)
participant_totals = participant_activity.groupby(["dataset", "User"], observed=False)["windows"].sum().reset_index(name="accepted_windows")
all_activities = sorted(metadata["label_from_npz"].unique())
participant_matrix = participant_activity.pivot_table(index=["dataset", "User"], columns="label_from_npz", values="windows", aggfunc="sum", fill_value=0).reindex(columns=all_activities, fill_value=0)
participant_matrix["accepted_windows"] = participant_matrix.sum(axis=1)
participant_matrix["missing_activities"] = (participant_matrix[all_activities] == 0).sum(axis=1)
participant_matrix["largest_activity_share"] = participant_matrix[all_activities].max(axis=1) / participant_matrix["accepted_windows"]
participant_matrix.reset_index().to_csv(TABLE_DIR / "participant_coverage_and_imbalance.csv", index=False)
display(participant_totals)
display(participant_matrix.reset_index())

ax = participant_activity.pivot_table(index="User", columns="label_from_npz", values="windows", aggfunc="sum", fill_value=0).plot.bar(figsize=(12, 5), title="Activity counts by participant")
ax.set_xlabel("Participant")
ax.set_ylabel("Accepted windows")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "participant_activity_grouped.png", dpi=160)
plt.show()

share = participant_matrix[all_activities].div(participant_matrix["accepted_windows"], axis=0)
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(share.to_numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(all_activities)), all_activities, rotation=45, ha="right")
ax.set_yticks(range(len(share)), [f"{d}/{u}" for d, u in share.index])
ax.set_title("Within-participant activity distribution")
ax.set_xlabel("Activity")
ax.set_ylabel("Dataset / participant")
fig.colorbar(im, ax=ax, label="Share of participant windows")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "participant_activity_heatmap.png", dpi=160)
plt.show()
print("Potential coverage concerns: rows with missing_activities > 0 or largest_activity_share >= 0.50.")
display(participant_matrix.reset_index().query("missing_activities > 0 or largest_activity_share >= 0.50"))

## Device differences

In [ ]:
device_activity = metadata.groupby(["dataset", "Model", "label_from_npz"], observed=False).size().rename("windows").reset_index()
device_activity.to_csv(TABLE_DIR / "device_activity_counts.csv", index=False)
display(device_activity)
for dataset_name, subset in device_activity.groupby("dataset", observed=False):
    pivot = subset.pivot(index="Model", columns="label_from_npz", values="windows").fillna(0)
    ax = pivot.plot.bar(figsize=(12, 5), title=f"Activity distribution across {dataset_name} device models")
    ax.set_xlabel("Device model")
    ax.set_ylabel("Accepted windows")
    ax.legend(title="Activity", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"device_activity_{dataset_name}.png", dpi=160)
    plt.show()
comparison = metadata.groupby(["Model", "dataset"], observed=False).size().unstack(fill_value=0).reset_index()
comparison.to_csv(TABLE_DIR / "phone_watch_device_comparison.csv", index=False)
display(comparison)
comparison.set_index("Model").plot.bar(figsize=(10, 4), title="Accepted windows by device model")
plt.xlabel("Device model")
plt.ylabel("Accepted windows")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phone_watch_device_comparison.png", dpi=160)
plt.show()

## Temporal coverage

In [ ]:
temporal_daily = metadata.groupby(["dataset", "start_day"], observed=False).size().rename("accepted_windows").reset_index()
temporal_daily.to_csv(TABLE_DIR / "accepted_windows_by_day.csv", index=False)
display(temporal_daily.head())
if metadata["start_datetime"].notna().any():
    daily = temporal_daily.pivot(index="start_day", columns="dataset", values="accepted_windows").fillna(0)
    daily.plot(figsize=(12, 4), marker="o", title="Accepted windows over recorded start dates")
    plt.xlabel("Recorded start date")
    plt.ylabel("Accepted windows")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "temporal_coverage_by_day.png", dpi=160)
    plt.show()
    temporal_activity = metadata.groupby(["dataset", "start_day", "label_from_npz"], observed=False).size().rename("windows").reset_index()
    temporal_activity.to_csv(TABLE_DIR / "activity_by_recorded_day.csv", index=False)
    temporal_plot = temporal_activity.pivot_table(index="start_day", columns="label_from_npz", values="windows", aggfunc="sum", fill_value=0)
    temporal_plot.plot(figsize=(12, 5), title="Recorded activity labels over time")
    plt.xlabel("Recorded start date")
    plt.ylabel("Accepted windows")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "activity_over_time.png", dpi=160)
    plt.show()
    print("Temporal activity plots use recorded labels only; no activity is inferred from timestamps.")
else:
    print("No parseable timestamps were available in the metadata.")

## Provenance and rejection diagnostics

The following records describe rejected or diagnostic processing events. They are not accepted windows and are never added to the accepted-window counts.

In [ ]:
def load_jsonl_records(root, filename):
    with gzip.open(root / filename, "rt", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

diagnostic_tables = {}
for name, root in DATASETS.items():
    records = load_jsonl_records(root, "rejections.jsonl.gz")
    diagnostic_tables[name] = pd.DataFrame(records)
    diagnostic_tables[name].insert(0, "dataset", name)
rejections = pd.concat(diagnostic_tables.values(), ignore_index=True)
rejection_reason = rejections.groupby(["dataset", "reason"], observed=False).size().rename("diagnostic_records").reset_index()
rejection_reason.to_csv(TABLE_DIR / "rejection_counts_by_reason.csv", index=False)
rejection_group = rejections.groupby(["dataset", "user", "model", "device", "reason"], observed=False).size().rename("diagnostic_records").reset_index()
rejection_group.to_csv(TABLE_DIR / "rejection_counts_by_group.csv", index=False)
rejection_sensor = rejections.groupby(["dataset", "sensor_segment", "reason"], observed=False).size().rename("diagnostic_records").reset_index()
rejection_sensor.to_csv(TABLE_DIR / "rejection_counts_by_sensor_segment.csv", index=False)
display(rejection_reason)
display(rejection_group.head(20))
rejection_reason.pivot(index="reason", columns="dataset", values="diagnostic_records").fillna(0).plot.barh(figsize=(10, 5), title="Diagnostic/rejection records by reason")
plt.xlabel("Diagnostic records")
plt.ylabel("Recorded reason")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "rejection_diagnostics_by_reason.png", dpi=160)
plt.show()
print("Accepted windows remain defined by the NPZ shard contents and stage2_summary.json; diagnostics are separate.")

## Example raw windows

Each example uses the documented six-channel order: `acc_x`, `acc_y`, `acc_z`, `gyro_x`, `gyro_y`, `gyro_z`. The arrays are plotted exactly as stored; no normalization or other transformation is applied.

In [ ]:
example_rows = []
for activity in sorted(metadata["label_from_npz"].unique()):
    candidates = metadata.loc[metadata["label_from_npz"].eq(activity)].sort_values(["User", "Model", "window_id"])
    chosen = candidates.drop_duplicates(["User", "Model"]).head(2)
    for _, row in chosen.iterrows():
        example_rows.append(row)

def locate_window(row):
    data = datasets[row["dataset"]]
    index = data["metadata"].index[data["metadata"]["window_id"].eq(row["window_id"])][0]
    return data["X"][index]

for number, row in enumerate(example_rows):
    window = locate_window(row)
    fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
    for channel_index, (axis, channel) in enumerate(zip(axes.flat, CHANNELS)):
        axis.plot(window[:, channel_index], linewidth=1)
        axis.set_title(channel)
        axis.set_ylabel("stored value")
        axis.set_xlabel("sample index")
    fig.suptitle(f"Raw 128-sample window: {row['label_from_npz']} | {row['dataset']} | participant {row['User']} | {row['Model']} ({row['Device']})")
    fig.tight_layout()
    filename = f"example_window_{number:02d}_{row['dataset']}_{row['label_from_npz']}.png"
    fig.savefig(FIGURE_DIR / filename, dpi=160)
    plt.show()
    print(f"Interpretation: recorded {row['label_from_npz']} example from participant {row['User']} and device model {row['Model']}; traces are descriptive only.")

## Basic sensor statistics

In [ ]:
stats_rows = []
for name, data in datasets.items():
    flattened = data["X"].reshape(-1, data["X"].shape[-1])
    for index, channel in enumerate(CHANNELS):
        values = flattened[:, index]
        stats_rows.append({"dataset": name, "channel": channel, "mean": np.mean(values), "std": np.std(values), "min": np.min(values), "max": np.max(values), "finite": np.isfinite(values).all(), "constant": np.ptp(values) == 0})
stats = pd.DataFrame(stats_rows)
stats.to_csv(TABLE_DIR / "per_channel_statistics.csv", index=False)
display(stats)
for sensor_name, channels in [("accelerometer", CHANNELS[:3]), ("gyroscope", CHANNELS[3:])]:
    subset = stats[stats["channel"].isin(channels)]
    subset.set_index(["dataset", "channel"])[["mean", "std", "min", "max"]].plot.bar(figsize=(10, 4), title=f"{sensor_name.title()} channel statistics")
    plt.ylabel("Stored value")
    plt.xlabel("Dataset / channel")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{sensor_name}_statistics.png", dpi=160)
    plt.show()
print("Constant channels:")
display(stats[stats["constant"]])
print("Non-finite channels:")
display(stats[~stats["finite"]])
print("Potential scale differences should be reviewed from the table; arrays were not modified.")

## Key observations

Use the generated CSV tables and figures to review class imbalance, participant and device coverage, phone-versus-watch differences, and data-quality diagnostics. In particular, check participants with missing activities or a dominant activity before any participant-safe evaluation. Rejection diagnostics describe processing exclusions and must not be interpreted as accepted-window counts. This notebook makes no claims about model accuracy and does not select preprocessing or model settings.